In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Figure 5.1 - Core Processing
与 fig2-1 同风格的 LUT 加载：Excel R/G/B -> ListedColormap，背景透明，由 mask 控制显示。
物理参数对齐 batch_downsampling_pipeline v1.2.0（仅 xy，2D 切片）。
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from scipy.ndimage import gaussian_filter, zoom

# =============================================================================
# Physical parameters
# =============================================================================
SPACING_IN_XY = (218/336, 166/256)   # (X, Y) mm
SPACING_OUT_XY = (1.8, 1.8)          # target CEST spacing (mm)
SIGMA_ADD_MM_XY = (0.713, 0.713)     # MPRAGE sigma_add_mm (probability labels use linear)
NUM_CLASSES_DEFAULT = 102

SIGMA_ADD_PX = (
    SIGMA_ADD_MM_XY[0] / SPACING_IN_XY[0],
    SIGMA_ADD_MM_XY[1] / SPACING_IN_XY[1],
)

ZOOM_FACTORS = (
    SPACING_IN_XY[0] / SPACING_OUT_XY[0],  # y-direction (pipeline X)
    SPACING_IN_XY[1] / SPACING_OUT_XY[1],  # x-direction (pipeline Y)
    1.0                                    # channels
)

# =============================================================================
# LUT helper (fig2-1 style)
# =============================================================================
def load_colormap(excel_path: str):
    """直接从 Excel 加载 102 类配色，背景透明 (alpha=0)。"""
    try:
        df = pd.read_excel(excel_path, engine='openpyxl') if excel_path.endswith((".xlsx", ".xls")) else pd.read_csv(excel_path, sep=None, engine="python")
        valid_labels = pd.to_numeric(df['one_hot_loc_alex_label'], errors='coerce').dropna()
        max_label = int(valid_labels.max()) + 10
        colors = np.zeros((max_label, 4))  # RGBA
        colors[0] = [0, 0, 0, 0]  # 背景透明
        for _, row in df.iterrows():
            lbl_val = row.get('one_hot_loc_alex_label')
            if pd.isna(lbl_val) or str(lbl_val).strip() == '[]':
                continue
            idx = int(lbl_val)
            r = row['R'] / 255.0
            g = row['G'] / 255.0
            b = row['B'] / 255.0
            colors[idx] = [r, g, b, 1.0]
        return mcolors.ListedColormap(colors)
    except Exception as e:
        print(f"⚠️ 无法读取 Excel 文件 ({e})，使用 tab20 代替。")
        return plt.cm.tab20

def lut_to_rgb_table(cmap: mcolors.ListedColormap, num_classes: int) -> np.ndarray:
    """Extract RGB (drop alpha) from ListedColormap/tab20 for mixing."""
    table = np.zeros((num_classes, 3), dtype=float)
    for i in range(num_classes):
        rgba = cmap(i)
        table[i] = rgba[:3]
    return table

def lut_name_raw(label: int, lut_df: pd.DataFrame | None) -> str:
    """Return raw tissue name from LUT; no cleaning."""
    if lut_df is None:
        return f"Label {label}"
    row = lut_df[lut_df['one_hot_loc_alex_label'] == label]
    if row.empty:
        return f"Label {label}"
    for k in ('freesurfer_tissue_name', 'tissue_name', 'col label'):
        if k in row.columns:
            val = row[k].iloc[0]
            if pd.notna(val):
                return str(val)
    return f"Label {label}"

# =============================================================================
# Core processing
# =============================================================================
def _normalize_prob(p: np.ndarray) -> np.ndarray:
    s = p.sum(axis=-1, keepdims=True)
    s = np.where(s > 0, s, 1.0)
    return p / s

def one_hot_encode(labels_2d: np.ndarray, num_classes: int) -> np.ndarray:
    h, w = labels_2d.shape
    one_hot = np.zeros((h, w, num_classes), dtype=np.float32)
    for c in range(num_classes):
        mask = (labels_2d == c)
        if mask.any():
            one_hot[..., c] = mask.astype(np.float32)
    return one_hot

def process_roi(roi_labels: np.ndarray, cmap: mcolors.ListedColormap, num_classes: int | None = None):
    max_label = int(np.max(roi_labels))
    num_classes = num_classes or max(NUM_CLASSES_DEFAULT, max_label + 1)

    one_hot = one_hot_encode(roi_labels, num_classes)

    soft_hr = gaussian_filter(one_hot,
                              sigma=(SIGMA_ADD_PX[0], SIGMA_ADD_PX[1], 0.0),
                              mode="constant")
    soft_hr = _normalize_prob(soft_hr)

    soft_lr = zoom(soft_hr, zoom=ZOOM_FACTORS, order=1, mode="reflect")
    soft_lr = _normalize_prob(soft_lr)

    colors_rgb = lut_to_rgb_table(cmap, num_classes)
    return soft_hr.astype(np.float32), soft_lr.astype(np.float32), colors_rgb


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Visualization helpers (mask-aware coloring)
"""

def pick_mixed_voxel(proba_lr: np.ndarray, mask_lr: np.ndarray):
    entropy = -np.sum(proba_lr * np.log(proba_lr + 1e-8), axis=-1)
    entropy = np.where(mask_lr > 0.5, entropy, -np.inf)
    y, x = np.unravel_index(np.argmax(entropy), entropy.shape)
    return int(y), int(x), proba_lr[y, x]

def prob_to_rgb(prob_map: np.ndarray, colors: np.ndarray) -> np.ndarray:
    return np.tensordot(prob_map, colors, axes=([2], [0]))

def plot_figure_5_1(roi_anatomy: np.ndarray, roi_labels: np.ndarray, roi_mask: np.ndarray,\
                    cmap: mcolors.ListedColormap, lut_df: pd.DataFrame | None, overrides: dict | None = None):
    assert roi_anatomy.shape == roi_labels.shape == roi_mask.shape, "anatomy/labels/mask shape mismatch"

    soft_hr, soft_lr, colors_rgb = process_roi(roi_labels, cmap)
    num_classes = colors_rgb.shape[0]

    h_hr, w_hr = roi_labels.shape
    width_mm = w_hr * SPACING_IN_XY[1]
    height_mm = h_hr * SPACING_IN_XY[0]

    mask_lr = zoom(roi_mask.astype(float), zoom=ZOOM_FACTORS[:2], order=0, mode='nearest')
    mask_lr = (mask_lr > 0.5).astype(float)

    rgb_hr = prob_to_rgb(soft_hr, colors_rgb)
    rgb_lr = prob_to_rgb(soft_lr, colors_rgb)
    rgb_hr[roi_mask == 0] = 0.0
    rgb_lr[mask_lr == 0] = 0.0

    # rotate 180° for display
    rot2 = lambda x: np.rot90(x, 2) if x.ndim == 2 else np.rot90(x, 2, axes=(0, 1))
    roi_anatomy_r = rot2(roi_anatomy)
    roi_labels_r = rot2(roi_labels)
    roi_mask_r = rot2(roi_mask)
    rgb_hr_r = rot2(rgb_hr)
    rgb_lr_r = rot2(rgb_lr)
    mask_lr_r = rot2(mask_lr)
    soft_lr_r = rot2(soft_lr)

    vy, vx, voxel_prob = pick_mixed_voxel(soft_lr_r, mask_lr_r)
    h_lr, w_lr, _ = soft_lr_r.shape
    voxel_w_mm = width_mm / w_lr
    voxel_h_mm = height_mm / h_lr

    fig = plt.figure(figsize=(18, 4.8), facecolor="white")
    # preserve aspect ratio of panel A for all: use same height; compute width from first image
    h, w = roi_anatomy_r.shape
    base_aspect = w / h if h > 0 else 1.0
    gs = gridspec.GridSpec(1, 4, width_ratios=[base_aspect, base_aspect, base_aspect, base_aspect], wspace=0.18)

    # Panel A
    axA = fig.add_subplot(gs[0])
    axA.imshow(roi_anatomy_r, cmap="gray", origin="upper")
    masked_lbl = np.ma.masked_where(roi_mask_r == 0, roi_labels_r)
    axA.imshow(masked_lbl, cmap=cmap, origin="upper", alpha=0.6, interpolation="nearest")
        # Panel A: 强调这是 MPRAGE 网格上的硬标签
    axA.set_title("A. Hard Parcellation\n(MPRAGE Grid)", fontweight="bold")
    axA.axis("off")

    # Panel B
    axB = fig.add_subplot(gs[1])
    axB.imshow(roi_anatomy_r, cmap="gray", origin="upper", alpha=0.25)
    axB.imshow(rgb_hr_r, origin="upper")
    # Panel B: 强调这是对部分容积效应（Partial Volume）的物理近似
    axB.set_title("B. Partial Volume Approximation\n(Physical Space Smoothing)", fontweight="bold")
    axB.axis("off")

    # Panel C
    axC = fig.add_subplot(gs[2])
    extent = [0, width_mm, height_mm, 0]
    axC.imshow(rgb_lr_r, origin="upper", interpolation="nearest", extent=extent)
    for gx in np.arange(0, width_mm + 1e-6, SPACING_OUT_XY[1]):
        axC.axvline(gx, color="white", lw=0.6, alpha=0.9)
    for gy in np.arange(0, height_mm + 1e-6, SPACING_OUT_XY[0]):
        axC.axhline(gy, color="white", lw=0.6, alpha=0.9)
    rect = plt.Rectangle((vx * voxel_w_mm, vy * voxel_h_mm), voxel_w_mm, voxel_h_mm,
                         edgecolor="cyan", facecolor="none", lw=2.0)
    axC.add_patch(rect)
    # Panel C: 强调这是最终的 Native CEST 网格
    axC.set_title("C. Resampled to Native CEST\n(1.8mm × 1.8mm × 3.0mm)", fontweight="bold")
    axC.set_xticks([]); axC.set_yticks([])

    # Panel D
    axD = fig.add_subplot(gs[3])
    top_idx = np.argsort(voxel_prob)[::-1][:4]
    raw_names = [lut_name_raw(int(i), lut_df) for i in top_idx]
    print("Top voxel labels (raw):", dict(zip(top_idx.tolist(), raw_names)))
    raw_names = [lut_name_raw(int(i), lut_df) for i in top_idx]
    print("Top voxel labels (raw):", dict(zip(top_idx.tolist(), raw_names)))
    top_vals = voxel_prob[top_idx]
    bar_colors = colors_rgb[top_idx]
    names = [overrides.get(int(i), lut_name_raw(int(i), lut_df)) if overrides else lut_name_raw(int(i), lut_df) for i in top_idx]
    axD.bar(range(len(top_idx)), top_vals, color=bar_colors, edgecolor="k")
    axD.set_xticks(range(len(top_idx)))
    axD.set_xticklabels(names, rotation=20, ha="right")
    axD.set_ylim(0, 1.0)
    axD.set_ylabel("Probability")
    # Panel D: 关键修改！将 "Signature" 改为 "Label" 或 "Target"
    # 原文: "Soft Voxel Signature" -> 容易混淆Input和Output
    # 修改: "Soft Label Target" -> 明确这是 Ground Truth
    axD.set_title("D. Per-Voxel Soft Label Target\n(Probabilistic Supervision $q$)", fontweight="bold")



    plt.tight_layout()
    return fig


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Execution: load MAT + LUT -> slice -> mask-crop -> PSF soft labels -> plot
"""

import h5py
from pathlib import Path

DATA_PATH = Path("/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat")
LUT_PATH = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx")
OUTPUT_FIG = Path("Figure_5_1_per_voxel_soft_labels.png")
SLICE_IDX = 222
ANATOMY_CHANNEL = 341
CROP_MARGIN = 20
MANUAL_NAME_OVERRIDES = {
    # --- Subcortical & General Structures ---
    0: "Background",
    4: "Cerebellum Cortex",
    6: "Caudate",
    7: "Putamen",
    8: "Pallidum",
    11: "Brain Stem",
    13: "Amygdala",
    14: "CSF (Cerebrospinal Fluid)",
    16: "Substantia Nigra",
    18: "Blood Vessel",
    19: "Choroid Plexus",
    21: "Optic Chiasm",
    97: "Internal Capsule (Anterior)",
    98: "Internal Capsule (Posterior)",

    # --- Cortex (Grey Matter) ---
    28: "Unknown Cortex",
    31: "Caudal Middle Frontal Gyrus",
    34: "Fusiform Gyrus",
    36: "Inferior Temporal Gyrus",
    37: "Isthmus Cingulate",
    38: "Lateral Occipital Cortex",
    40: "Lingual Gyrus",
    41: "Medial Orbitofrontal Cortex",
    44: "Paracentral Lobule",
    45: "Pars Opercularis",
    46: "Pars Orbitalis",
    47: "Pars Triangularis",
    50: "Posterior Cingulate",
    51: "Precentral Gyrus",
    52: "Precuneus",
    53: "Rostral Anterior Cingulate",
    55: "Superior Frontal Gyrus",
    56: "Superior Parietal Cortex",
    57: "Superior Temporal Gyrus",
    58: "Supramarginal Gyrus",
    59: "Frontal Pole",
    60: "Transverse Temporal Gyrus",
    62: "Temporal Pole",

    # --- White Matter (Subcortical) ---
    63: "Unknown White Matter",
    66: "Caudal Middle Frontal WM",
    68: "Fusiform WM",
    70: "Inferior Temporal WM",
    71: "Isthmus Cingulate WM",
    72: "Lateral Occipital WM",
    73: "Lateral Orbitofrontal WM",  # Note: 73 was missing in ctx list but present in wm list
    74: "Lingual WM",
    75: "Medial Orbitofrontal WM",
    79: "Pars Opercularis WM",
    80: "Pars Orbitalis WM",
    81: "Pars Triangularis WM",
    84: "Posterior Cingulate WM",
    85: "Precentral WM",
    86: "Precuneus WM",
    87: "Rostral Anterior Cingulate WM",
    89: "Superior Frontal WM",
    90: "Superior Parietal WM",
    91: "Superior Temporal WM",
    92: "Supramarginal WM",
    93: "Frontal Pole WM",
    94: "Transverse Temporal WM",
    95: "Insula WM",
    96: "Unsegmented White Matter"
}
def load_data_labels_mask(mat_path: Path):
    print(f"Loading data: {mat_path} ...")
    with h5py.File(mat_path, "r") as f:
        if "data" not in f:
            raise ValueError("MAT 文件缺少 data")
        data = f["data"][:]
        if data.shape[0] in [341, 351]:
            data = np.moveaxis(data, 0, -1)
        if "region_labels" in f:
            labels = f["region_labels"][:]
        elif "one_hot_loc_alex_label" in f:
            oh = f["one_hot_loc_alex_label"][:]
            axis = 0 if oh.shape[0] == 102 else -1
            labels = np.argmax(oh, axis=axis)
        else:
            raise ValueError("MAT 文件缺少 region_labels/one_hot_loc_alex_label")
        if "region_mask" in f:
            mask = f["region_mask"][:]
        elif "region" in f:
            mask = f["region"][:]
        else:
            print("[Warn] 未找到 region_mask，使用 labels>0 作为掩码")
            mask = (labels > 0).astype(np.uint8)
    print(f"  data shape: {data.shape}, labels shape: {labels.shape}, mask shape: {mask.shape}")
    return data.astype(np.float32), labels.astype(int), mask.astype(np.uint8)

def crop_to_mask(mask2d: np.ndarray, margin: int = 0):
    ys, xs = np.nonzero(mask2d)
    if len(ys) == 0:
        return (0, mask2d.shape[0], 0, mask2d.shape[1])
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    y0 = max(0, y0 - margin); y1 = min(mask2d.shape[0], y1 + margin)
    x0 = max(0, x0 - margin); x1 = min(mask2d.shape[1], x1 + margin)
    return (y0, y1, x0, x1)

def run_figure_5_1():
    cmap = load_colormap(str(LUT_PATH))
    lut_df = pd.read_excel(LUT_PATH, engine='openpyxl') if LUT_PATH.suffix in ('.xlsx', '.xls') else None
    data, labels, mask = load_data_labels_mask(DATA_PATH)
    assert 0 <= SLICE_IDX < data.shape[0], "SLICE_IDX 越界"
    assert ANATOMY_CHANNEL < data.shape[-1], "ANATOMY_CHANNEL 越界"

    img_slice = data[SLICE_IDX, :, :, ANATOMY_CHANNEL]
    lbl_slice = labels[SLICE_IDX, :, :]
    msk_slice = mask[SLICE_IDX, :, :]

    y0, y1, x0, x1 = crop_to_mask(msk_slice, margin=CROP_MARGIN)
    img_roi = img_slice[y0:y1, x0:x1]
    lbl_roi = lbl_slice[y0:y1, x0:x1]
    msk_roi = msk_slice[y0:y1, x0:x1]

    # 打印 ROI 中出现的标签及名称，便于手工修改
    unique_lbls = np.unique(lbl_roi[msk_roi > 0]).astype(int)
    print("ROI labels:")
    for lbl in unique_lbls:
        raw_name = lut_name_raw(int(lbl), lut_df)
        manual = MANUAL_NAME_OVERRIDES.get(int(lbl))
        print(f"  {lbl}: {manual if manual else raw_name}")
    fig = plot_figure_5_1(img_roi, lbl_roi, msk_roi, cmap, lut_df, MANUAL_NAME_OVERRIDES)
    fig.savefig(OUTPUT_FIG, dpi=300)
    plt.close(fig)
    print(f"✅ Figure saved: {OUTPUT_FIG}")

run_figure_5_1()
